In [ ]:

# Food Waste Database Exploration
## Step 1: Database Connection
This notebook connects to the Food Waste PostgreSQL database running inside Docker using SQLAlchemy.


In [ ]:
# Step 2 — Install and Import Libraries
%pip install sqlalchemy psycopg2-binary pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy import (
    String,
    Integer,
    Numeric,
    ForeignKey,
    Identity,
    Table,
    Column,
    select,
    func
)
from decimal import Decimal
from getpass import getpass
import pandas as pd

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
#Step 3 — Connect to Docker PostgreSQL

Create a SQLAlchemy engine to connect to the PostgreSQL database running inside Docker.

In [ ]:


connection_url = URL.create(
    "postgresql+psycopg2",
    username="airflow",
    password="airflow",
    host="127.0.0.1",
    port=5433,
    database="food_waste_db"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            version(),
            inet_server_addr(),
            inet_server_port(),
            current_database()
    """))

    row = result.fetchone()

    print("Version:", row[0])
    print("Server Address:", row[1])
    print("Server Port:", row[2])
    print("Database:", row[3])

print("\nDocker PostgreSQL connection successful!")


Version: PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit
Server Address: 172.18.0.3
Server Port: 5432
Database: food_waste_db

Docker PostgreSQL connection successful!


In [ ]:
## Step 4 — Inspect Database Schemas and Tables

Explore the available schemas and identify all tables inside the food_waste schema.

In [ ]:


inspector = inspect(engine)

print("Available Schemas:")
print(inspector.get_schema_names())

print("\nTables in food_waste schema:")

tables = inspector.get_table_names(schema="food_waste")

for table in tables:
    print(f"- {table}")

print(f"\nTotal Tables: {len(tables)}")


Available Schemas:
['food_waste', 'information_schema', 'public']

Tables in food_waste schema:
- country
- stg_activity_long
- stg_food_waste
- activity
- observation_activity
- observation
- commodity
- method
- source

Total Tables: 9


In [ ]:
## Step 5 — Inspect Table Structures

Inspect the columns, data types, primary keys, and foreign keys of all tables in the food_waste schema.

In [7]:
for table_name in tables:
    print(f"\n{'=' * 60}")
    print(f"Table: {table_name}")
    print("=" * 60)

    columns = inspector.get_columns(table_name, schema="food_waste")

    for column in columns:
        print(
            f"Column: {column['name']:<25} "
            f"Type: {str(column['type']):<20} "
            f"Nullable: {column['nullable']}"
        )

    primary_key = inspector.get_pk_constraint(
        table_name,
        schema="food_waste"
    )

    print("\nPrimary Key:", primary_key["constrained_columns"])

    foreign_keys = inspector.get_foreign_keys(
        table_name,
        schema="food_waste"
    )

    if foreign_keys:
        print("Foreign Keys:")
        for fk in foreign_keys:
            print(
                f"  {fk['constrained_columns']} -> "
                f"{fk['referred_table']}.{fk['referred_columns']}"
            )
    else:
        print("Foreign Keys: None")


Table: country
Column: m49_code                  Type: VARCHAR(3)           Nullable: False
Column: country_name              Type: VARCHAR(150)         Nullable: False

Primary Key: ['m49_code']
Foreign Keys: None

Table: stg_activity_long
Column: record_id                 Type: TEXT                 Nullable: True
Column: activity                  Type: TEXT                 Nullable: True

Primary Key: []
Foreign Keys: None

Table: stg_food_waste
Column: m49_code                  Type: TEXT                 Nullable: True
Column: country                   Type: TEXT                 Nullable: True
Column: cpc_code                  Type: TEXT                 Nullable: True
Column: commodity                 Type: TEXT                 Nullable: True
Column: year                      Type: TEXT                 Nullable: True
Column: loss_percentage           Type: TEXT                 Nullable: True
Column: activity                  Type: TEXT                 Nullable: True
Column: method_

In [ ]:
## Step 6 — Inspect Foreign Key Relationships

Explore the relationships between the core tables using their foreign key constraints.

In [8]:
for table_name in tables:
    foreign_keys = inspector.get_foreign_keys(
        table_name,
        schema="food_waste"
    )

    if foreign_keys:
        print(f"\nTable: {table_name}")

        for fk in foreign_keys:
            print(f"  Foreign Key: {fk['constrained_columns']}")
            print(f"  References: {fk['referred_table']}.{fk['referred_columns']}")
            print(f"  On Delete: {fk.get('options', {}).get('ondelete', 'NO ACTION')}")


Table: observation_activity
  Foreign Key: ['activity_id']
  References: activity.['activity_id']
  On Delete: NO ACTION
  Foreign Key: ['record_id']
  References: observation.['record_id']
  On Delete: CASCADE

Table: observation
  Foreign Key: ['cpc_code']
  References: commodity.['cpc_code']
  On Delete: NO ACTION
  Foreign Key: ['m49_code']
  References: country.['m49_code']
  On Delete: NO ACTION
  Foreign Key: ['method_id']
  References: method.['method_id']
  On Delete: NO ACTION
  Foreign Key: ['source_id']
  References: source.['source_id']
  On Delete: NO ACTION


In [ ]:
## Step 7 — Define SQLAlchemy ORM Models

Define the database structure using SQLAlchemy's Declarative ORM mapping.

The models represent the seven core tables and their relationships.
The two staging tables will remain available for raw data inspection.

In [10]:

class Base(DeclarativeBase):
    pass


observation_activity = Table(
    "observation_activity",
    Base.metadata,
    Column(
        "record_id",
        Integer,
        ForeignKey("food_waste.observation.record_id", ondelete="CASCADE"),
        primary_key=True
    ),
    Column(
        "activity_id",
        Integer,
        ForeignKey("food_waste.activity.activity_id"),
        primary_key=True
    ),
    schema="food_waste"
)


class Country(Base):
    __tablename__ = "country"
    __table_args__ = {"schema": "food_waste"}

    m49_code: Mapped[str] = mapped_column(String(3), primary_key=True)
    country_name: Mapped[str] = mapped_column(String(150), unique=True, nullable=False)

    observations: Mapped[list["Observation"]] = relationship(back_populates="country")


class Commodity(Base):
    __tablename__ = "commodity"
    __table_args__ = {"schema": "food_waste"}

    cpc_code: Mapped[str] = mapped_column(String(30), primary_key=True)
    commodity_name: Mapped[str] = mapped_column(String(250), unique=True, nullable=False)

    observations: Mapped[list["Observation"]] = relationship(back_populates="commodity")


class Method(Base):
    __tablename__ = "method"
    __table_args__ = {"schema": "food_waste"}

    method_id: Mapped[int] = mapped_column(Integer, Identity(), primary_key=True)
    method_name: Mapped[str] = mapped_column(String(250), unique=True, nullable=False)

    observations: Mapped[list["Observation"]] = relationship(back_populates="method")


class Source(Base):
    __tablename__ = "source"
    __table_args__ = {"schema": "food_waste"}

    source_id: Mapped[int] = mapped_column(Integer, Identity(), primary_key=True)
    reference: Mapped[str | None] = mapped_column(String, nullable=True)
    url: Mapped[str | None] = mapped_column(String, nullable=True)

    observations: Mapped[list["Observation"]] = relationship(back_populates="source")


class Activity(Base):
    __tablename__ = "activity"
    __table_args__ = {"schema": "food_waste"}

    activity_id: Mapped[int] = mapped_column(Integer, Identity(), primary_key=True)
    activity_name: Mapped[str] = mapped_column(String(250), unique=True, nullable=False)

    observations: Mapped[list["Observation"]] = relationship(
        secondary=observation_activity,
        back_populates="activities"
    )


class Observation(Base):
    __tablename__ = "observation"
    __table_args__ = {"schema": "food_waste"}

    record_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    year: Mapped[int] = mapped_column(Integer, nullable=False)
    loss_percentage: Mapped[Decimal] = mapped_column(Numeric(8, 3), nullable=False)

    cpc_code: Mapped[str] = mapped_column(
        String(30),
        ForeignKey("food_waste.commodity.cpc_code"),
        nullable=False
    )

    method_id: Mapped[int | None] = mapped_column(
        Integer,
        ForeignKey("food_waste.method.method_id"),
        nullable=True
    )

    m49_code: Mapped[str] = mapped_column(
        String(3),
        ForeignKey("food_waste.country.m49_code"),
        nullable=False
    )

    source_id: Mapped[int] = mapped_column(
        Integer,
        ForeignKey("food_waste.source.source_id"),
        nullable=False
    )

    country: Mapped["Country"] = relationship(back_populates="observations")
    commodity: Mapped["Commodity"] = relationship(back_populates="observations")
    method: Mapped["Method | None"] = relationship(back_populates="observations")
    source: Mapped["Source"] = relationship(back_populates="observations")

    activities: Mapped[list["Activity"]] = relationship(
        secondary=observation_activity,
        back_populates="observations"
    )


print("All 7 ORM models defined successfully!")


All 7 ORM models defined successfully!


In [ ]:
## Step 8 — Verify ORM Mapping

Verify that all seven ORM models are correctly mapped to their corresponding database tables and inspect their columns.

In [11]:
models = [
    Country,
    Commodity,
    Method,
    Source,
    Activity,
    Observation
]

print("ORM Model Mapping:")

for model in models:
    print(f"\nModel: {model.__name__}")
    print(f"Table: {model.__table__.fullname}")

    for column in model.__table__.columns:
        print(f"  - {column.name}: {column.type}")

print("\nAssociation Table:")
print(observation_activity.fullname)

print("\nORM mapping verification completed!")

ORM Model Mapping:

Model: Country
Table: food_waste.country
  - m49_code: VARCHAR(3)
  - country_name: VARCHAR(150)

Model: Commodity
Table: food_waste.commodity
  - cpc_code: VARCHAR(30)
  - commodity_name: VARCHAR(250)

Model: Method
Table: food_waste.method
  - method_id: INTEGER
  - method_name: VARCHAR(250)

Model: Source
Table: food_waste.source
  - source_id: INTEGER
  - reference: VARCHAR
  - url: VARCHAR

Model: Activity
Table: food_waste.activity
  - activity_id: INTEGER
  - activity_name: VARCHAR(250)

Model: Observation
Table: food_waste.observation
  - record_id: INTEGER
  - year: INTEGER
  - loss_percentage: NUMERIC(8, 3)
  - cpc_code: VARCHAR(30)
  - method_id: INTEGER
  - m49_code: VARCHAR(3)
  - source_id: INTEGER

Association Table:
food_waste.observation_activity

ORM mapping verification completed!


In [ ]:
## Step 9 — Verify ORM Relationships

Inspect the relationships defined between the ORM models, including one-to-many and many-to-many relationships.

In [12]:
relationship_models = [
    Country,
    Commodity,
    Method,
    Source,
    Activity,
    Observation
]

for model in relationship_models:
    print(f"\n{model.__name__}:")

    for relationship_name in model.__mapper__.relationships.keys():
        print(f"  - {relationship_name}")

print("\nRelationship verification completed!")


Country:
  - observations

Commodity:
  - observations

Method:
  - observations

Source:
  - observations

Activity:
  - observations

Observation:
  - country
  - commodity
  - method
  - source
  - activities

Relationship verification completed!


In [ ]:
## Step 10 — Query Database Using ORM

Use SQLAlchemy ORM Session to retrieve records and verify that the database contains the expected data.

In [13]:
with Session(engine) as session:

    print("Database Records Count")
    print("=" * 40)

    print("Countries:", session.scalar(
        select(func.count()).select_from(Country)
    ))

    print("Commodities:", session.scalar(
        select(func.count()).select_from(Commodity)
    ))

    print("Methods:", session.scalar(
        select(func.count()).select_from(Method)
    ))

    print("Sources:", session.scalar(
        select(func.count()).select_from(Source)
    ))

    print("Activities:", session.scalar(
        select(func.count()).select_from(Activity)
    ))

    print("Observations:", session.scalar(
        select(func.count()).select_from(Observation)
    ))

    print("Observation Activities:", session.scalar(
        select(func.count()).select_from(observation_activity)
    ))

Database Records Count
Countries: 142
Commodities: 196
Methods: 11
Sources: 1216
Activities: 52
Observations: 31164
Observation Activities: 40436


In [ ]:
## Step 11 — Verify ORM Relationships with Real Data

Retrieve an observation and access its related country, commodity, method, source, and activities through SQLAlchemy ORM relationships.

In [14]:
with Session(engine) as session:

    observation = session.scalars(
        select(Observation).limit(1)
    ).first()

    if observation:

        print("Observation Details")
        print("=" * 40)

        print("Record ID:", observation.record_id)
        print("Year:", observation.year)
        print("Loss Percentage:", observation.loss_percentage)

        print("\nRelated Country:")
        print(observation.country.country_name)

        print("\nRelated Commodity:")
        print(observation.commodity.commodity_name)

        print("\nRelated Method:")
        if observation.method:
            print(observation.method.method_name)
        else:
            print("Not Available")

        print("\nRelated Source:")
        print("Reference:", observation.source.reference)
        print("URL:", observation.source.url)

        print("\nRelated Activities:")
        for activity in observation.activities:
            print("-", activity.activity_name)

    else:
        print("No observations found.")

Observation Details
Record ID: 1
Year: 2015
Loss Percentage: 5.880

Related Country:
Myanmar

Related Commodity:
Rice, milled

Related Method:
Controlled Experiment

Related Source:
Reference: Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama; Grant R Singleton, (2015), Estimating Rodent Losses To Stored Rice As A Means To Assess Efficacy Of Rodent Management
URL: None

Related Activities:
- Storage


In [ ]:
## Step 12 — Filter Observations Using ORM

Use SQLAlchemy ORM select() and where() to retrieve observations for a specific country.

In [15]:
with Session(engine) as session:

    statement = (
        select(Observation)
        .join(Observation.country)
        .where(Country.country_name == "Myanmar")
        .limit(10)
    )

    observations = session.scalars(statement).all()

    print(f"Observations for Myanmar: {len(observations)}")
    print("=" * 60)

    for observation in observations:
        print(
            f"Record ID: {observation.record_id} | "
            f"Year: {observation.year} | "
            f"Loss: {observation.loss_percentage}% | "
            f"Commodity: {observation.commodity.commodity_name}"
        )

Observations for Myanmar: 10
Record ID: 1 | Year: 2015 | Loss: 5.880% | Commodity: Rice, milled
Record ID: 2 | Year: 2015 | Loss: 11.770% | Commodity: Rice, milled
Record ID: 3 | Year: 2015 | Loss: 17.650% | Commodity: Rice, milled
Record ID: 4 | Year: 2015 | Loss: 3.570% | Commodity: Rice, milled
Record ID: 5 | Year: 2015 | Loss: 5.350% | Commodity: Rice, milled
Record ID: 6 | Year: 2015 | Loss: 1.780% | Commodity: Rice, milled
Record ID: 7 | Year: 2009 | Loss: 5.220% | Commodity: Groundnuts, excluding shelled
Record ID: 8 | Year: 2008 | Loss: 5.430% | Commodity: Groundnuts, excluding shelled
Record ID: 9 | Year: 2007 | Loss: 5.610% | Commodity: Groundnuts, excluding shelled
Record ID: 10 | Year: 2006 | Loss: 5.400% | Commodity: Groundnuts, excluding shelled


In [ ]:
## Step 13 — Calculate Average Food Loss by Country

Use SQLAlchemy aggregation functions and GROUP BY to calculate the average food loss percentage for each country.

In [16]:
with Session(engine) as session:

    statement = (
        select(
            Country.country_name,
            func.count(Observation.record_id).label("total_observations"),
            func.avg(Observation.loss_percentage).label("average_loss")
        )
        .join(Country.observations)
        .group_by(Country.m49_code, Country.country_name)
        .order_by(func.avg(Observation.loss_percentage).desc())
        .limit(10)
    )

    results = session.execute(statement).all()

    print("Top 10 Countries by Average Food Loss")
    print("=" * 65)

    for country_name, total_observations, average_loss in results:
        print(
            f"{country_name:<30} "
            f"Observations: {total_observations:<6} "
            f"Average Loss: {average_loss:.3f}%"
        )

Top 10 Countries by Average Food Loss
Haiti                          Observations: 6      Average Loss: 39.500%
Gabon                          Observations: 1      Average Loss: 35.000%
Oman                           Observations: 1      Average Loss: 35.000%
Saint Kitts and Nevis          Observations: 1      Average Loss: 30.000%
Bahrain                        Observations: 1      Average Loss: 25.000%
Algeria                        Observations: 1      Average Loss: 25.000%
Trinidad and Tobago            Observations: 9      Average Loss: 22.944%
Jamaica                        Observations: 15     Average Loss: 21.227%
Peru                           Observations: 286    Average Loss: 20.818%
Australia                      Observations: 43     Average Loss: 18.402%


In [ ]:
## Step 14 — Calculate Average Food Loss by Commodity

Calculate the average food loss percentage and total observations for each commodity using SQLAlchemy ORM aggregation.

In [17]:
with Session(engine) as session:

    statement = (
        select(
            Commodity.commodity_name,
            func.count(Observation.record_id).label("total_observations"),
            func.avg(Observation.loss_percentage).label("average_loss")
        )
        .join(Commodity.observations)
        .group_by(Commodity.cpc_code, Commodity.commodity_name)
        .order_by(func.avg(Observation.loss_percentage).desc())
        .limit(10)
    )

    results = session.execute(statement).all()

    print("Top 10 Commodities by Average Food Loss")
    print("=" * 75)

    for commodity_name, total_observations, average_loss in results:
        print(
            f"{commodity_name:<45} "
            f"Observations: {total_observations:<6} "
            f"Average Loss: {average_loss:.3f}%"
        )

Top 10 Commodities by Average Food Loss
Snails, fresh, chilled, frozen, dried, salted or in brine, except sea snails Observations: 2      Average Loss: 50.000%
Vegetables, dehydrated                        Observations: 2      Average Loss: 43.300%
Grapefruit juice                              Observations: 21     Average Loss: 41.891%
Orange juice                                  Observations: 20     Average Loss: 40.300%
Pineapple juice                               Observations: 20     Average Loss: 40.016%
Canned mushrooms                              Observations: 20     Average Loss: 34.000%
Other legumes, for forage                     Observations: 1      Average Loss: 26.700%
Apple juice                                   Observations: 19     Average Loss: 26.667%
Bovine meat, salted, dried or smoked          Observations: 2      Average Loss: 25.400%
Meat of sheep, fresh or chilled               Observations: 13     Average Loss: 25.135%


In [ ]:
## Step 15 — Data Quality Validation

Validate the observations table by checking missing values, invalid food loss percentages, duplicate record IDs, and foreign key integrity.

In [18]:
with Session(engine) as session:

    total_records = session.scalar(
        select(func.count()).select_from(Observation)
    )

    missing_values = session.execute(
        select(
            func.count().filter(Observation.year.is_(None)),
            func.count().filter(Observation.loss_percentage.is_(None)),
            func.count().filter(Observation.cpc_code.is_(None)),
            func.count().filter(Observation.m49_code.is_(None)),
            func.count().filter(Observation.source_id.is_(None))
        )
    ).one()

    invalid_loss = session.scalar(
        select(func.count())
        .select_from(Observation)
        .where(
            (Observation.loss_percentage < 0) |
            (Observation.loss_percentage > 100)
        )
    )

    duplicate_record_ids = session.scalar(
        select(
            func.count(Observation.record_id) -
            func.count(func.distinct(Observation.record_id))
        )
        .select_from(Observation)
    )

    missing_country = session.scalar(
        select(func.count())
        .select_from(Observation)
        .outerjoin(Observation.country)
        .where(Country.m49_code.is_(None))
    )

    missing_commodity = session.scalar(
        select(func.count())
        .select_from(Observation)
        .outerjoin(Observation.commodity)
        .where(Commodity.cpc_code.is_(None))
    )

    missing_source = session.scalar(
        select(func.count())
        .select_from(Observation)
        .outerjoin(Observation.source)
        .where(Source.source_id.is_(None))
    )

    print("Data Quality Validation")
    print("=" * 50)

    print("Total Observations:", total_records)

    print("\nMissing Values:")
    columns = [
        "year",
        "loss_percentage",
        "cpc_code",
        "m49_code",
        "source_id"
    ]

    for column, count in zip(columns, missing_values):
        print(f"{column}: {count}")

    print("\nInvalid Loss Percentages:", invalid_loss)
    print("Duplicate Record IDs:", duplicate_record_ids)

    print("\nMissing Relationships:")
    print("Country:", missing_country)
    print("Commodity:", missing_commodity)
    print("Source:", missing_source)

Data Quality Validation
Total Observations: 31164

Missing Values:
year: 0
loss_percentage: 0
cpc_code: 0
m49_code: 0
source_id: 0

Invalid Loss Percentages: 0
Duplicate Record IDs: 0

Missing Relationships:
Country: 0
Commodity: 0
Source: 0


In [ ]:
## Step 16 — Validate Observation-Activity Relationships

Validate the many-to-many relationship table by checking duplicate combinations and missing references to observations or activities.

In [19]:
with Session(engine) as session:

    total_links = session.scalar(
        select(func.count()).select_from(observation_activity)
    )

    duplicate_links = session.scalar(
        select(
            func.count() -
            func.count(
                func.distinct(
                    func.concat(
                        observation_activity.c.record_id,
                        "-",
                        observation_activity.c.activity_id
                    )
                )
            )
        )
        .select_from(observation_activity)
    )

    missing_observations = session.scalar(
        select(func.count())
        .select_from(observation_activity)
        .outerjoin(
            Observation,
            observation_activity.c.record_id == Observation.record_id
        )
        .where(Observation.record_id.is_(None))
    )

    missing_activities = session.scalar(
        select(func.count())
        .select_from(observation_activity)
        .outerjoin(
            Activity,
            observation_activity.c.activity_id == Activity.activity_id
        )
        .where(Activity.activity_id.is_(None))
    )

    print("Observation-Activity Validation")
    print("=" * 50)

    print("Total Relationships:", total_links)
    print("Duplicate Relationships:", duplicate_links)
    print("Missing Observations:", missing_observations)
    print("Missing Activities:", missing_activities)

Observation-Activity Validation
Total Relationships: 40436
Duplicate Relationships: 0
Missing Observations: 0
Missing Activities: 0


In [ ]:
## Step 17 — Inspect Staging Tables

Inspect the staging tables using SQLAlchemy reflection and verify their record counts.

In [20]:
from sqlalchemy import MetaData, Table

metadata = MetaData()

stg_food_waste = Table(
    "stg_food_waste",
    metadata,
    schema="food_waste",
    autoload_with=engine
)

stg_activity_long = Table(
    "stg_activity_long",
    metadata,
    schema="food_waste",
    autoload_with=engine
)

with engine.connect() as connection:

    food_waste_count = connection.execute(
        select(func.count()).select_from(stg_food_waste)
    ).scalar_one()

    activity_count = connection.execute(
        select(func.count()).select_from(stg_activity_long)
    ).scalar_one()

print("Staging Tables Record Counts")
print("=" * 45)

print("stg_food_waste:", food_waste_count)
print("stg_activity_long:", activity_count)

Staging Tables Record Counts
stg_food_waste: 31164
stg_activity_long: 40436


In [ ]:
## Step 18 — Compare Staging and Final Tables

Compare the record counts of staging and final tables to verify that the data loading process preserved all records.

In [21]:
with engine.connect() as connection:

    counts = {
        "stg_food_waste": connection.execute(
            select(func.count()).select_from(stg_food_waste)
        ).scalar_one(),

        "observation": connection.execute(
            select(func.count()).select_from(Observation.__table__)
        ).scalar_one(),

        "stg_activity_long": connection.execute(
            select(func.count()).select_from(stg_activity_long)
        ).scalar_one(),

        "observation_activity": connection.execute(
            select(func.count()).select_from(observation_activity)
        ).scalar_one()
    }

print("Staging vs Final Tables")
print("=" * 55)

comparisons = [
    ("stg_food_waste", "observation"),
    ("stg_activity_long", "observation_activity")
]

for staging_table, final_table in comparisons:

    staging_count = counts[staging_table]
    final_count = counts[final_table]

    status = "MATCH" if staging_count == final_count else "MISMATCH"

    print(f"\n{staging_table}: {staging_count}")
    print(f"{final_table}: {final_count}")
    print(f"Status: {status}")

Staging vs Final Tables

stg_food_waste: 31164
observation: 31164
Status: MATCH

stg_activity_long: 40436
observation_activity: 40436
Status: MATCH


In [ ]:
## Step 19 — Final Database Summary

### Database Information
- Database: `food_waste_db`
- Schema: `food_waste`
- DBMS: PostgreSQL 16
- ORM: SQLAlchemy 2.0 Style

### Database Tables

| Table | Records |
|---|---:|
| country | 142 |
| commodity | 196 |
| method | 11 |
| activity | 52 |
| source | 1,216 |
| observation | 31,164 |
| observation_activity | 40,436 |
| stg_food_waste | 31,164 |
| stg_activity_long | 40,436 |

### Validation Results
- All expected record counts verified.
- No missing values in required observation columns.
- No invalid loss percentages.
- No duplicate observation IDs.
- No duplicate observation-activity relationships.
- All tested foreign key relationships are valid.
- Staging and final table record counts match.

### Conclusion
The Food Waste PostgreSQL database was successfully inspected and validated using SQLAlchemy ORM. All nine tables contain the expected number of records, the defined relationships work correctly, and the performed data quality checks passed.